# DJIA LangGraph Track Tuner

Automated track analysis and parameter tuning using LangGraph multi-agent orchestration.

**Features:**
- Single-track analysis and tuning
- Batch processing over multiple tracks
- Iterative parameter optimization
- Quality evaluation and feedback
- Comprehensive reporting

## Architecture

```
Single Track Agent:
  LoadTrack → Initialize → Analyze → Evaluate → [Tune or Finalize]

Batch Agent:
  [Track 1] → [Track 2] → [Track 3] → ... → Aggregate Results
```

## Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

from src.ai.track_tuner_graph import run_single_track, run_batch_tracks
from src.dsp.phrasing_engine import time_to_bar

print("[SETUP] LangGraph Track Tuner loaded")

[SETUP] LangGraph Track Tuner loaded


## Example 1: Analyze Single Track

In [2]:
# Find Marrakech track
import glob

files = glob.glob("data/**/*.mp3", recursive=True)
marrakech = [f for f in files if "marrakech" in f.lower()]

if marrakech:
    track_path = marrakech[0]
    print(f"Found: {track_path}")
else:
    print("Marrakech track not found")
    track_path = files[0] if files else None
    print(f"Using instead: {track_path}")

Found: data\ Hermanez - Marrakech.mp3


In [3]:
# Run single track analysis with minimal config
print("Starting single track analysis...\n")

result = run_single_track(
    track_path=track_path,
    preset="minimal",  # Use minimal preset
    max_iterations=3   # Max 3 tuning iterations
)

# Display results
print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print(f"Track: {result.get('track_name', 'unknown')}")
print(f"BPM: {result.get('bpm', 0):.2f}")
print(f"Duration: {result.get('duration', 0):.1f}s ({result.get('duration', 0)/60:.2f}m)")
print(f"\nFinal Segmentation:")
print(f"  Segments: {result.get('current_quality', {}).get('num_segments', 0)}")
print(f"  Avg bars/segment: {result.get('current_quality', {}).get('avg_bars_per_segment', 0):.1f}")
print(f"  Regularity (std): {result.get('current_quality', {}).get('regularity_std', 0):.1f}")
print(f"  Quality score: {result.get('current_quality', {}).get('quality_score', 0):.2f}")
print(f"\nTuning:")
print(f"  Satisfied: {result.get('satisfied', False)}")
print(f"  Iterations: {result.get('iterations_completed', 0)}")
print(f"  Reason: {result.get('reason', 'N/A')}")
print(f"\nFinal Config:")
cfg = result.get('current_config', {})
print(f"  Novelty threshold: {cfg.get('novelty_threshold', 0):.2f}")
print(f"  Min segment duration: {cfg.get('min_segment_duration', 0):.1f}s")
print(f"  Breakdown threshold: {cfg.get('breakdown_duration_threshold', 0):.1f}s")

Starting single track analysis...


ANALYSIS COMPLETE
Track:  Hermanez - Marrakech
BPM: 129.20
Duration: 635.2s (10.59m)

Final Segmentation:
  Segments: 2
  Avg bars/segment: 171.0
  Regularity (std): 40.5
  Quality score: 0.82

Tuning:
  Satisfied: True
  Iterations: 0
  Reason: Quality threshold met

Final Config:
  Novelty threshold: 0.65
  Min segment duration: 12.0s
  Breakdown threshold: 32.0s


In [4]:
# Display segments
print("\n" + "="*80)
print("SEGMENT DETAILS")
print("="*80)

segments = result.get('current_segments', [])
print(f"{'#':>2} | {'Label':25} | {'Start Bar':>12} | {'End Bar':>12} | {'Bars':>8}")
print("-" * 80)

for i, seg in enumerate(segments, 1):
    label = seg['label'].split('(')[0].strip()
    bars = seg['end_bar'] - seg['start_bar']
    print(f"{i:2} | {label:25} | {seg['start_bar']:12.1f} | {seg['end_bar']:12.1f} | {bars:8.1f}")


SEGMENT DETAILS
 # | Label                     |    Start Bar |      End Bar |     Bars
--------------------------------------------------------------------------------
 1 | drop                      |          0.0 |        211.5 |    211.5
 2 | drop                      |        211.5 |        342.0 |    130.5


In [5]:
# Display analysis history
print("\n" + "="*80)
print("TUNING HISTORY")
print("="*80)

history = result.get('analysis_history', [])
print(f"{'Iter':>4} | {'Quality':>8} | {'Reason':30}")
print("-" * 60)

for entry in history:
    print(f"{entry['iteration']:4} | {entry['quality_score']:8.2f} | {entry['reason'][:30]:30}")


TUNING HISTORY
Iter |  Quality | Reason                        
------------------------------------------------------------
   0 |     0.82 | Quality threshold met         


## Example 2: Batch Process Multiple Tracks

In [6]:
# Get list of all tracks
import glob

all_tracks = sorted(glob.glob("data/**/*.mp3", recursive=True))
print(f"Found {len(all_tracks)} tracks")
print(f"\nFirst 5 tracks:")
for i, t in enumerate(all_tracks[:5], 1):
    print(f"  {i}. {Path(t).stem}")

Found 71 tracks

First 5 tracks:
  1.  Hermanez - Marrakech
  2. 01 - Gaiser - Pullpush
  3. 01 - ambivalent - nineteen
  4. 01 - larsson - need the sun - siberia
  5. 01 - markus fix - research - bside


In [ ]:
# Run batch analysis on first 5 tracks
print(f"\nRunning batch analysis on {min(5, len(all_tracks))} tracks...\n")

batch_result = run_batch_tracks(
    track_paths=all_tracks[:5],
    preset="minimal"
)

print("\nBatch processing complete!")


Running batch analysis on 5 tracks...



In [ ]:
# Summarize batch results
results = batch_result.get('results', [])

print("\n" + "="*100)
print("BATCH ANALYSIS SUMMARY")
print("="*100)

if results:
    # Create DataFrame
    df = pd.DataFrame([
        {
            'Track': r['track_name'],
            'BPM': f"{r['bpm']:.1f}",
            'Segments': r['num_segments'],
            'Avg Bars': f"{r['avg_bars']:.1f}",
            'Quality': f"{r['quality_score']:.2f}",
            'Satisfied': 'Yes' if r['satisfied'] else 'No',
            'Iterations': r['iterations_used']
        }
        for r in results
    ])

    print(df.to_string(index=False))

    print(f"\nSummary:")
    satisfied = sum(1 for r in results if r['satisfied'])
    print(f"  Total tracks: {len(results)}")
    print(f"  Satisfied: {satisfied}/{len(results)}")
    print(f"  Avg quality: {np.mean([r['quality_score'] for r in results]):.2f}")
    print(f"  Total iterations: {sum(r['iterations_used'] for r in results)}")
else:
    print("No results yet")

## Example 3: Compare Different Presets

In [ ]:
# Analyze same track with different presets
presets = ["default", "minimal", "house", "techno"]
results_by_preset = {}

print("Testing different presets on same track...\n")

for preset in presets:
    print(f"Running {preset}...", end=" ")
    result = run_single_track(
        track_path=track_path,
        preset=preset,
        max_iterations=1  # Just one iteration for comparison
    )
    results_by_preset[preset] = result
    print("Done")

print("\n" + "="*80)
print("PRESET COMPARISON")
print("="*80)

comparison_df = pd.DataFrame([
    {
        'Preset': preset,
        'Segments': result.get('current_quality', {}).get('num_segments', 0),
        'Avg Bars': f"{result.get('current_quality', {}).get('avg_bars_per_segment', 0):.1f}",
        'Regularity': f"{result.get('current_quality', {}).get('regularity_std', 0):.1f}",
        'Quality': f"{result.get('current_quality', {}).get('quality_score', 0):.2f}",
        'Threshold': f"{result.get('current_config', {}).get('novelty_threshold', 0):.2f}",
    }
    for preset, result in results_by_preset.items()
])

print(comparison_df.to_string(index=False))

## Example 4: Detailed Segment Analysis

In [ ]:
# Get best result (highest quality)
best_preset = max(results_by_preset.items(), key=lambda x: x[1].get('current_quality', {}).get('quality_score', 0))[0]
best_result = results_by_preset[best_preset]

print(f"Best result: {best_preset} preset")
print(f"Quality score: {best_result.get('current_quality', {}).get('quality_score', 0):.2f}\n")

# Detailed segment breakdown
segments = best_result.get('current_segments', [])
bpm = best_result.get('bpm', 120)

print("="*100)
print("DETAILED SEGMENT BREAKDOWN")
print("="*100)
print(f"{'#':>2} | {'Label':25} | {'Time Range':20} | {'Start Bar':>12} | {'End Bar':>12} | {'Bars':>8}")
print("-" * 100)

for i, seg in enumerate(segments, 1):
    label = seg['label'].split('(')[0].strip()
    bars = seg['end_bar'] - seg['start_bar']
    time_range = f"{seg['start_time']:.1f}-{seg['end_time']:.1f}s"
    print(f"{i:2} | {label:25} | {time_range:20} | {seg['start_bar']:12.1f} | {seg['end_bar']:12.1f} | {bars:8.1f}")

## Example 5: Export Results

In [ ]:
# Export analysis results to CSV
import json

# Create detailed export
export_data = {
    'track_name': best_result.get('track_name'),
    'bpm': best_result.get('bpm'),
    'duration': best_result.get('duration'),
    'preset_used': best_preset,
    'quality_score': best_result.get('current_quality', {}).get('quality_score'),
    'num_segments': best_result.get('current_quality', {}).get('num_segments'),
    'satisfied': best_result.get('satisfied'),
    'segments': best_result.get('current_segments', []),
    'config': best_result.get('current_config', {}),
}

# Save to file
output_path = Path('results') / f"{best_result.get('track_name', 'track')}_analysis.json"
output_path.parent.mkdir(exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"Results exported to: {output_path}")
print(f"\nExport contents:")
print(json.dumps(export_data, indent=2))

## Agent Workflow Summary

### Single Track Agent Flow

1. **LoadTrack** → Extract BPM, duration, basic info
2. **InitializeConfig** → Load preset parameters
3. **AnalyzeTrack** → Extract segments with current config
4. **EvaluateQuality** → Score segmentation (0-1)
5. **Decision Point:**
   - If quality >= 0.70 → **Finalize**
   - If max iterations reached → **Finalize**
   - Otherwise → **SuggestTuning** → (back to AnalyzeTrack)

### Tuning Strategy

- **Too many segments?** → Increase novelty threshold & min duration
- **False breakdowns?** → Increase breakdown duration threshold
- **Irregular distribution?** → Decrease novelty threshold for better detection

### Batch Processing

- Process tracks sequentially
- Each track gets up to 3 tuning iterations
- Results aggregated and exported
- Quality metrics tracked per track